# Role 2 — GNN (Arindam)
Direction-aware GraphSAGE on sparse adjacency, with a SIGN fallback. Follows the playbook contracts.

1. Run cells 1–3 now: they test everything on a toy cross-bank graph.
2. When Vishwajeet announces `data/processed_sample/`, run section B with `SAMPLE = True`.
3. Overnight, run section B again with `SAMPLE = False`.

In [1]:
import sys
!{sys.executable} -m pip install -q torch_geometric


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))   # shared/ -> import mulenet
import numpy as np, pandas as pd, torch
from mulenet.gnn import run_setting, save_scores, quick_report, train_model, benchmark_inference
from mulenet.toy import make_toy
torch.set_num_threads(3)
print('torch', torch.__version__)

torch 2.11.0+cpu


## A. Toy test (works before real data exists)
Expected: combined PR-AUC clearly above silo, and joined-graph-own-labels in between.

In [3]:
nodes_t, lv_t = make_toy()
res = {}
for setting, scope in [('silo','all'), ('combined','all'), ('combined','own')]:
    s, _ = run_setting(lv_t, nodes_t, setting, kind='sage', label_scope=scope, verbose=False)
    res[f'{setting}/{scope}'] = quick_report(s, nodes_t, budget=0.02).iloc[-1]
pd.DataFrame(res).T[['mules','caught','recall_at_budget','pr_auc']]

[silo/all] member B1: 1116 own train accounts
[silo/all] member B2: 892 own train accounts
[silo/all] member B3: 702 own train accounts
[silo/all] member B4: 538 own train accounts
[silo/all] member B5: 351 own train accounts
[combined] training on all members' labels
[combined/own] member B1: 1116 own train accounts
[combined/own] member B2: 892 own train accounts
[combined/own] member B3: 702 own train accounts
[combined/own] member B4: 538 own train accounts
[combined/own] member B5: 351 own train accounts


,mules,caught,recall_at_budget,pr_auc
silo/all,120,25,0.208333,0.851992
combined/all,120,25,0.208333,0.957424
combined/own,120,25,0.208333,0.922454


## B. Real data (IBM AML HI-Small)
Needs Vishwajeet's `mulenet/common.py` and `data/processed_sample/` (or `data/processed/`).

Ask him to also return `edge_index_dir` (original sender→receiver edges, duplicates merged) from `load_view`, so the model keeps fan-in and fan-out apart. Without it the model still runs, treating edges as undirected.

In [3]:
SAMPLE = True          # False for the full overnight run
KIND = 'sage'          # 'sign' = faster fallback if 'sage' is too slow
from mulenet.common import load_view
proc = '../data/processed_sample' if SAMPLE else '../data/processed'
nodes = pd.read_csv(f'{proc}/nodes.csv').sort_values('node_idx').reset_index(drop=True)
assert (nodes.node_idx.values == np.arange(len(nodes))).all(), 'nodes.csv must be in node_idx order'
out = '../results/scores_sample' if SAMPLE else '../results/scores'
print(len(nodes), 'accounts |', nodes.label.sum(), 'mules |', nodes.inst.value_counts().to_dict())

510772 accounts | 1680 mules | {'B2': 105675, 'B5': 105220, 'B4': 104001, 'B3': 103881, 'B1': 91995}


Post in the team chat before running the next cell on full data (`SAMPLE = False`): it is the heavy job.

In [4]:
t0 = time.time()
all_info = []
for setting, scope, name in [('combined','all','gnn_combined'), ('silo','all','gnn_silo')]:
    s, info = run_setting(load_view, nodes, setting, kind=KIND, label_scope=scope, sample=SAMPLE, verbose=True)
    save_scores(s, f'{out}/{name}.csv')
    all_info += [{**i, 'name': name} for i in info]
    print(quick_report(s, nodes, budget=0.01).to_string(index=False))
pd.DataFrame(all_info).to_csv(f'{out}/gnn_training_log.csv', index=False)
print(f'done in {(time.time()-t0)/60:.1f} min')

[combined] training on all members' labels
  epoch   1  loss 1.5045  val PR-AUC 0.0062
  epoch  10  loss 0.9517  val PR-AUC 0.0174
  epoch  20  loss 0.8062  val PR-AUC 0.0450
  epoch  30  loss 0.7321  val PR-AUC 0.0424
  best epoch 19  val PR-AUC 0.0461  (291.0s)
wrote ../results/scores_sample/gnn_combined.csv (510772 rows)
inst  test_accts  mules  alerts  caught  recall_at_budget   pr_auc
  B1       18275     53     183       6          0.113208 0.024527
  B2       21183     64     212      14          0.218750 0.048710
  B3       20803     86     208      18          0.209302 0.068337
  B4       20739     71     207      12          0.169014 0.051133
  B5       21155     62     212      10          0.161290 0.067040
 ALL      102155    336    1022      60          0.178571 0.041188
[silo/all] member B1: 55244 own train accounts
  epoch   1  loss 1.5198  val PR-AUC 0.0062
  epoch  10  loss 0.8875  val PR-AUC 0.0151
  epoch  20  loss 0.7312  val PR-AUC 0.0264
  epoch  30  loss 0.6171  

## C. IBM Z inference benchmark (for Vipul's slide)
Only measured numbers go on slides.

In [ ]:
data = load_view('combined', sample=SAMPLE)
model, _, info = train_model(data, data.train_mask, data.val_mask, kind=KIND, verbose=False)
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), f'../models/gnn_{KIND}_combined{"_sample" if SAMPLE else ""}.pt')
bench = benchmark_inference(model, data, kind=KIND, runs=5)
bench.update({'machine': 'IBM LinuxONE Community Cloud, s390x, 3 vCPU, 8 GB', 'sample': SAMPLE})
os.makedirs('../results', exist_ok=True)
json.dump(bench, open('../results/inference_benchmark.json', 'w'), indent=2)
bench

## D. Save to GitHub
Clear outputs first (Edit → Clear All Outputs), then over SSH:
```bash
cd ~/shared && git pull --rebase
git add mulenet/gnn.py mulenet/toy.py notebooks/20_gnn_arindam.ipynb
git commit -m "GNN: direction-aware GraphSAGE + SIGN, silo/combined runner" --author="Arindam Gogoi <arindamgogoi.cc2023@gmail.com>"
git push
```

In [5]:
from mulenet.evaluate import results_table
from mulenet.common import load_nodes
S = '../results/scores_sample'
nodes = load_nodes(sample=True)
rd = lambda n: pd.read_csv(f'{S}/{n}.csv', dtype={'acct_key': str})
t = results_table({'Logistic': (rd('lr_silo'), rd('lr_combined')),
                   'XGBoost':  (rd('xgb_silo'), rd('xgb_combined')),
                   'GNN (SIGN)': (rd('gnn_silo'), rd('gnn_combined'))}, nodes)
t[['model','recall_silo','recall_combined','uplift_pts','uplift_ci_low','uplift_ci_high','pr_auc_silo','pr_auc_combined']].round(3)

,model,recall_silo,recall_combined,uplift_pts,uplift_ci_low,uplift_ci_high,pr_auc_silo,pr_auc_combined
0,Logistic,0.179,0.173,-0.006,-0.024,0.012,0.034,0.036
1,XGBoost,0.179,0.223,0.045,0.009,0.083,0.027,0.050
2,GNN (SIGN),0.188,0.179,-0.009,-0.054,0.036,0.043,0.041


In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from snapml import GraphFeaturePreprocessor   # proves Snap ML imports
print('Snap ML OK')

from mulenet.common import load_nodes
from mulenet.gfp import smoke_test
smoke_test(sample=True)

Snap ML OK
params accepted: {'num_threads': 3, 'time_window': -1, 'vertex_stats': False, 'fan': True, 'fan_tw': 1440, 'fan_bins': [2, 3, 4, 6, 10], 'degree': True, 'degree_tw': 1440, 'degree_bins': [2, 3, 4, 6, 10], 'scatter-gather': True, 'scatter-gather_tw': 360, 'scatter-gather_bins': [2, 3, 4, 6], 'temp-cycle': True, 'temp-cycle_tw': 1440, 'temp-cycle_bins': [2, 3, 4, 6], 'lc-cycle': False}
input (200000, 4) -> output (200000, 32) in 1.09s (183,736 edges/s)
non-zero share per graph feature: [0.023, 0.003, 0.001, 0.0, 0.001, 0.031, 0.004, 0.001, 0.0, 0.019, 0.183, 0.0, 0.116, 0.053, 0.018, 0.17, 0.0, 0.098, 0.053, 0.044, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
node features: (510772, 56)


In [3]:
from mulenet.gfp import run_gfp_xgb
OUT = '../results/scores_sample'
run_gfp_xgb('silo', sample=True, out_dir=OUT)
run_gfp_xgb('combined', sample=True, out_dir=OUT)
run_gfp_xgb('combined', sample=True, out_dir=OUT, label_scope='own')

/home/ibm-user/shared/mulenet/common.py:150: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  edge_index_dir=ei_dir, y=torch.from_numpy(nodes["label"].to_numpy(np.int64)),


  Snap ML GFP on B1: 654,596 edges in 287.4s
[gfp-xgb/silo/all] view B1, labels B1: 55,244 train accounts, 75 features
  Snap ML GFP on B2: 656,943 edges in 23.5s
[gfp-xgb/silo/all] view B2, labels B2: 63,181 train accounts, 75 features
  Snap ML GFP on B3: 661,042 edges in 27.1s
[gfp-xgb/silo/all] view B3, labels B3: 62,445 train accounts, 75 features
  Snap ML GFP on B4: 661,054 edges in 21.7s
[gfp-xgb/silo/all] view B4, labels B4: 62,626 train accounts, 75 features
  Snap ML GFP on B5: 659,347 edges in 20.0s
[gfp-xgb/silo/all] view B5, labels B5: 62,967 train accounts, 75 features
wrote ../results/scores_sample/gfpxgb_silo.csv
  Snap ML GFP on combined: 2,076,752 edges in 514.8s
[gfp-xgb/combined/all] view combined, labels all: 306,463 train accounts, 75 features
wrote ../results/scores_sample/gfpxgb_combined.csv
[gfp-xgb/combined/own] view combined, labels B1: 55,244 train accounts, 75 features
[gfp-xgb/combined/own] view combined, labels B2: 63,181 train accounts, 75 features
[gfp

,acct_key,inst,split,score
0,10232_80117DA10,B1,test,0.108099
1,10232_803BDD920,B1,train,0.116652
2,10232_803BDF9B0,B1,val,0.087719
3,10232_803BE4BD0,B1,val,0.372832
4,10232_803BF8C30,B1,test,0.091295
...,...,...,...,...
510767,9691_8136FAD50,B5,train,0.044498
510768,9691_8137B2060,B5,val,0.037337
510769,9691_8140814B0,B5,train,0.507530
510770,9691_81417A610,B5,train,0.037337


In [4]:
import numpy as np
from mulenet.evaluate import results_table
from mulenet.common import load_nodes

G = np.load('../data/processed_sample/x_gfp_combined.npy')
print('non-zero share per node feature:', np.round((G != 0).mean(0), 3).tolist())

OUT = '../results/scores_sample'
nodes = load_nodes(sample=True)
rd = lambda n: pd.read_csv(f'{OUT}/{n}.csv', dtype={'acct_key': str})
t = results_table({'XGBoost': (rd('xgb_silo'), rd('xgb_combined')),
                   'GNN (SIGN)': (rd('gnn_silo'), rd('gnn_combined')),
                   'Snap ML graph patterns + XGBoost': (rd('gfpxgb_silo'), rd('gfpxgb_combined')),
                   'Graph patterns, own labels only': (rd('gfpxgb_silo'), rd('gfpxgb_combined_ownlabels'))}, nodes)
t[['model','recall_silo','recall_combined','uplift_pts','uplift_ci_low','uplift_ci_high','pr_auc_silo','pr_auc_combined']].round(3)

non-zero share per node feature: [0.336, 0.145, 0.049, 0.027, 0.041, 0.159, 0.08, 0.045, 0.026, 0.009, 0.294, 0.288, 0.275, 0.264, 0.202, 0.28, 0.203, 0.181, 0.143, 0.095, 0.0, 0.0, 0.0, 0.0, 0.001, 0.0, 0.0, 0.0, 0.32, 0.064, 0.013, 0.005, 0.002, 0.283, 0.231, 0.181, 0.144, 0.143, 0.401, 0.335, 0.298, 0.234, 0.111, 0.268, 0.266, 0.275, 0.286, 0.319, 0.0, 0.0, 0.0, 0.0, 0.001, 0.0, 0.0, 0.0]


,model,recall_silo,recall_combined,uplift_pts,uplift_ci_low,uplift_ci_high,pr_auc_silo,pr_auc_combined
0,XGBoost,0.179,0.223,0.045,0.009,0.083,0.027,0.050
1,GNN (SIGN),0.188,0.179,-0.009,-0.054,0.036,0.043,0.041
2,Snap ML graph patterns + XGBoost,0.176,0.241,0.065,0.027,0.101,0.032,0.055
3,"Graph patterns, own labels only",0.176,0.214,0.039,0.006,0.074,0.032,0.030


In [1]:
import os, sys, numpy as np, pandas as pd
sys.path.insert(0, os.path.abspath('..'))
from mulenet.gfp import run_gfp_xgb
from mulenet.evaluate import results_table
from mulenet.common import load_nodes

OUT = '../results/scores_sample'
for setting, scope in [('silo', 'all'), ('combined', 'all'), ('combined', 'own')]:
    run_gfp_xgb(setting, sample=True, out_dir=OUT, label_scope=scope)

import glob
G = np.load(sorted(glob.glob('../data/processed_sample/x_gfp_combined_*.npy'))[-1])
print('scatter-gather / cycle columns non-zero:', np.round((G[:, 20:28] != 0).mean(0), 3).tolist())

nodes = load_nodes(sample=True)
rd = lambda n: pd.read_csv(f'{OUT}/{n}.csv', dtype={'acct_key': str})
t = results_table({'XGBoost': (rd('xgb_silo'), rd('xgb_combined')),
                   'Snap ML graph patterns + XGBoost': (rd('gfpxgb_silo'), rd('gfpxgb_combined')),
                   'Graph patterns, own labels only': (rd('gfpxgb_silo'), rd('gfpxgb_combined_ownlabels'))}, nodes)
t[['model','recall_silo','recall_combined','uplift_pts','uplift_ci_low','uplift_ci_high','pr_auc_silo','pr_auc_combined']].round(3)

  Snap ML GFP on B1: 654,596 edges in 453.7s
[gfp-xgb/silo/all] view B1, labels B1: 55,244 train accounts, 75 features
  Snap ML GFP on B2: 656,943 edges in 29.4s
[gfp-xgb/silo/all] view B2, labels B2: 63,181 train accounts, 75 features
  Snap ML GFP on B3: 661,042 edges in 34.0s
[gfp-xgb/silo/all] view B3, labels B3: 62,445 train accounts, 75 features
  Snap ML GFP on B4: 661,054 edges in 28.7s
[gfp-xgb/silo/all] view B4, labels B4: 62,626 train accounts, 75 features
  Snap ML GFP on B5: 659,347 edges in 26.1s
[gfp-xgb/silo/all] view B5, labels B5: 62,967 train accounts, 75 features
wrote ../results/scores_sample/gfpxgb_silo.csv
  Snap ML GFP on combined: 2,076,752 edges in 665.5s
[gfp-xgb/combined/all] view combined, labels all: 306,463 train accounts, 75 features
wrote ../results/scores_sample/gfpxgb_combined.csv
[gfp-xgb/combined/own] view combined, labels B1: 55,244 train accounts, 75 features
[gfp-xgb/combined/own] view combined, labels B2: 63,181 train accounts, 75 features
[gfp

,model,recall_silo,recall_combined,uplift_pts,uplift_ci_low,uplift_ci_high,pr_auc_silo,pr_auc_combined
0,XGBoost,0.179,0.223,0.045,0.009,0.083,0.027,0.050
1,Snap ML graph patterns + XGBoost,0.220,0.280,0.060,0.018,0.104,0.030,0.067
2,"Graph patterns, own labels only",0.220,0.205,-0.015,-0.051,0.024,0.030,0.038


In [1]:
import os, sys, time
sys.path.insert(0, os.path.abspath('..'))
from mulenet.baselines import run_baseline
OUT = '../results/scores'
t0 = time.time()
for kind in ['lr', 'xgb']:
    for setting in ['silo', 'combined']:
        run_baseline(kind, setting, sample=False, out_dir=OUT)
print(f'BASELINES DONE in {(time.time() - t0) / 60:.0f} min')

[lr/silo] B1: trained on 55,652 accounts (622 mules)
[lr/silo] B2: trained on 63,860 accounts (755 mules)
[lr/silo] B3: trained on 62,868 accounts (812 mules)
[lr/silo] B4: trained on 63,010 accounts (834 mules)
[lr/silo] B5: trained on 63,662 accounts (791 mules)
wrote ../results/scores/lr_silo.csv
[lr/combined] combined: trained on 309,052 accounts (3814 mules)
wrote ../results/scores/lr_combined.csv
[xgb/silo] B1: trained on 55,652 accounts (622 mules)
[xgb/silo] B2: trained on 63,860 accounts (755 mules)
[xgb/silo] B3: trained on 62,868 accounts (812 mules)
[xgb/silo] B4: trained on 63,010 accounts (834 mules)
[xgb/silo] B5: trained on 63,662 accounts (791 mules)
wrote ../results/scores/xgb_silo.csv
[xgb/combined] combined: trained on 309,052 accounts (3814 mules)
wrote ../results/scores/xgb_combined.csv
BASELINES DONE in 1 min


In [ ]:
from mulenet.gfp import run_gfp_xgb
t0 = time.time()
for setting, scope in [('silo', 'all'), ('combined', 'all'), ('combined', 'own')]:
    run_gfp_xgb(setting, sample=False, out_dir=OUT, label_scope=scope)
print(f'ALL DONE in {(time.time() - t0) / 60:.0f} min')

  Snap ML GFP on B1: 1,734,103 edges in 865.4s
[gfp-xgb/silo/all] view B1, labels B1: 55,652 train accounts, 75 features
  Snap ML GFP on B2: 1,706,031 edges in 66.5s
[gfp-xgb/silo/all] view B2, labels B2: 63,860 train accounts, 75 features
  Snap ML GFP on B3: 1,722,826 edges in 71.3s
[gfp-xgb/silo/all] view B3, labels B3: 62,868 train accounts, 75 features
  Snap ML GFP on B4: 1,721,008 edges in 59.2s
[gfp-xgb/silo/all] view B4, labels B4: 63,010 train accounts, 75 features
  Snap ML GFP on B5: 1,714,736 edges in 54.8s
[gfp-xgb/silo/all] view B5, labels B5: 63,662 train accounts, 75 features
wrote ../results/scores/gfpxgb_silo.csv
